In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    print(mode(data))
    return fig

**Make a file containing descriptions for each geneset**

In [81]:
# columns 5 and 6 are short and long descriptions
datadir = '/home/ctrader/region_similarity/data/msigdb_genesets/metadata/'
filenames = os.listdir(datadir)

geneset_descriptions = []
for filename in filenames:
    # load data
    f = open(datadir + filename, 'r')
    lines = f.readlines()
    f.close()
    
    if len(lines) > 0:
        # extract name and descriptions
        name = lines[0].strip('\n').split('\t')[1]
        short_description = lines[5].strip('\n').split('\t')[1]
        long_description = lines[6].strip('\n').split('\t')[1]

        # add to list of all genesets
        geneset_descriptions.append([name, short_description, long_description])
    else:
        geneset_descriptions.append([filename.strip('.tsv'), '', ''])
        
geneset_descriptions = np.array(geneset_descriptions)

In [82]:
# # save
# datadir = '/home/ctrader/region_similarity/data/msigdb_genesets/'
# out_f = datadir + 'h_c2_geneset_metadata_descriptions_CT20230530.tsv'

# np.savetxt(out_f, geneset_descriptions, fmt='%s', delimiter='\t')

Send full ranks file to Ian (

**Max scoring state per bin - new regions and scores**

In [2]:
filename = '/home/ctrader/region_similarity/data/rank_based/epilogos.scores.CT20230118.txt'
f = open(filename, 'r')
epi_data = []
for line in f.readlines():
    epi_data.append(line.strip().split('\t'))
f.close()
epi_data = np.array(epi_data)

In [3]:
# find the max state (paper order) for each bin
bin_argmax = np.argmax(epi_data[:, 3:].astype(float), axis=1)

In [4]:
# find number of nonzero states within each bin
num_nonzero_per_bin = np.count_nonzero(epi_data[:, 3:].astype(float), axis=1)

In [5]:
np.where(num_nonzero_per_bin==0)[0].shape

(0,)

In [ ]:
## setting argmax values in bins with 0 values to nan
# bin_argmax = np.where(num_nonzero_per_bin==0, np.nan, bin_argmax) - not needed

In [6]:
region_sums = np.sum(epi_data[:, 3:].astype(float), axis=1)
np.where(region_sums < 0)[0].shape

(0,)

In [7]:
# counting the number of bins in which top 2 values are tied
c = 0
for i in range(len(epi_data)):
    x = -np.sort(-epi_data[i][3:].astype(float))[:2]
    if ((x[0] == x[1]) and (x[0] > 0)):
        c += 1
c
# there are not that many non-zero ties we have to worry about

160

In [12]:
# count for argmax of each bin over entire genome when not all 0 values
genome_count = np.bincount(bin_argmax.astype(int))
genome_count

array([  77467,   41217,   61076,   18880,  414424, 1782896,   12909,
          8268,  239335,  159946,  355418,  206584, 1101532,   45656,
         15077,  237579, 1332040, 7991847])

In [13]:
# in paper order (ie Quiescent is last)
genome_expected = genome_count / np.sum(genome_count)
genome_expected

array([0.00549328, 0.00292275, 0.00433097, 0.0013388 , 0.02938729,
       0.12642724, 0.00091539, 0.00058629, 0.01697152, 0.01134196,
       0.02520311, 0.01464911, 0.07811092, 0.00323752, 0.00106913,
       0.016847  , 0.09445651, 0.56671121])

In [14]:
states = np.array(['TssA', 'TssFlnk', 'TssFlnkU', 'TssFlnkD', 'Tx', 'TxWk', 'EnhG1', 'EnhG2', 'EnhA1', 'EnhA2', 'EnhWk', 'ZNF_Rpts', 'Het', 'TssBiv', 'EnhBiv', 'ReprPC', 'ReprPCWk', 'Quies'])

In [15]:
outfile = '/home/ctrader/region_similarity/data/rank_based/epilogos.winningstateperbin.genomewide.counts.expectedratio.officialorder.CT20230118.tsv'
np.savetxt(outfile, np.vstack((np.vstack((states, genome_count)), genome_expected)), delimiter='\t', fmt='%s')

In [17]:
bin_argmax_str = bin_argmax.astype(int).astype(str)

In [18]:
winning_state_txt = np.array(['winning_state'] * len(epi_data))

In [19]:
epi_data = np.hstack((epi_data, winning_state_txt.reshape(-1, 1)))
epi_data = np.hstack((epi_data, bin_argmax_str.reshape(-1, 1)))

In [20]:
epi_data[0]

array(['chr1', '10000', '10200', '0.00000', '0.00024', '0.00000',
       '0.00000', '0.00000', '-0.00740', '0.00000', '0.00000', '0.00000',
       '-0.00315', '-0.00401', '-0.00496', '-0.01689', '0.01173',
       '0.00000', '-0.00414', '0.00000', '0.39662', 'winning_state', '17'],
      dtype='<U21')

In [21]:
epi_data[0, [0, 1, 2, -2, -1]]

array(['chr1', '10000', '10200', 'winning_state', '17'], dtype='<U21')

In [23]:
outfile = '/home/ctrader/region_similarity/data/rank_based/epilogos.winningstateperbin.officialorder.CT20230118.txt'
np.savetxt(outfile, epi_data[:, [0, 1, 2, -2, -1]], delimiter='\t', fmt='%s')

In [ ]:
"""
In terminal:
sort-bed epilogos.winningstateperbin.officialorder.CT20230118.txt > epilogos.winningstateperbin.officialorder.CT20230118.sorted.bed

In preprocessing file:
bedmap --echo --echo-map-score hg38.25kb.maxMean.CT20230118.sorted.bed epilogos.winningstateperbin.officialorder.CT20230118.sorted.bed > regions_25kb_CT20230118.epilogos.maxstateperbin.officialstateorder.allmapped.bed
"""

**Max scoring state per bin**

In [5]:
filename = '/home/ctrader/region_similarity/data/epilogos_scores_new_CT20220815.txt'
f = open(filename, 'r')
epi_data = []
for line in f.readlines():
    epi_data.append(line.strip().split('\t'))
f.close()
epi_data = np.array(epi_data)

In [3]:
# find the max state (paper order) for each bin
bin_argmax = np.argmax(epi_data[:, 3:].astype(float), axis=1)

In [4]:
# find number of nonzero states within each bin
num_nonzero_per_bin = np.count_nonzero(epi_data[:, 3:].astype(float), axis=1)

In [5]:
# setting argmax values in bins with 0 values to nan
bin_argmax = np.where(num_nonzero_per_bin==0, np.nan, bin_argmax)

In [6]:
region_sums = np.sum(epi_data[:, 3:].astype(float), axis=1)
np.where(region_sums < 0)[0].shape

(0,)

In [7]:
# counting the number of bins in which top 2 values are tied
c = 0
for i in range(len(epi_data)):
    x = -np.sort(-epi_data[i][3:].astype(float))[:2]
    if ((x[0] == x[1]) and (x[0] > 0)):
        c += 1
c
# there are not that many non-zero ties we have to worry about

115

In [8]:
# count for argmax of each bin over entire genome when not all 0 values
genome_count = np.bincount(bin_argmax[~np.isnan(bin_argmax)].astype(int))
genome_count

array([  77612,   41720,   61052,   18879,  413085, 1851323,   12667,
          8150,  241569,  166702,  369285,  206488, 1170114,   45612,
         14921,  237377, 1405117, 7760478])

In [20]:
# in paper order (ie Quiescent is last)
genome_expected = genome_count / np.sum(genome_count)
genome_expected

array([0.00550356, 0.00295841, 0.00432927, 0.00133873, 0.02929234,
       0.13127948, 0.00089823, 0.00057793, 0.01712994, 0.01182103,
       0.02618643, 0.01464231, 0.08297415, 0.0032344 , 0.00105807,
       0.01683268, 0.09963849, 0.55030456])

In [23]:
states = np.array(['TssA', 'TssFlnk', 'TssFlnkU', 'TssFlnkD', 'Tx', 'TxWk', 'EnhG1', 'EnhG2', 'EnhA1', 'EnhA2', 'EnhWk', 'ZNF_Rpts', 'Het', 'TssBiv', 'EnhBiv', 'ReprPC', 'ReprPCWk', 'Quies'])

In [27]:
outfile = '/home/ctrader/region_similarity/data/rank_based/epilogos.winningstateperbin.genomewide.counts.expectedratio.officialorder.CT20230103.tsv'
np.savetxt(outfile, np.vstack((np.vstack((states, genome_count)), genome_expected)), delimiter='\t', fmt='%s')

In [10]:
bin_argmax_str = bin_argmax.astype(int).astype(str)
bin_argmax_str = np.where(np.isnan(bin_argmax), 'nan', bin_argmax_str)

In [11]:
bin_argmax_str[50]

'17'

In [13]:
bin_argmax_str

array(['nan', 'nan', 'nan', ..., 'nan', 'nan', 'nan'], dtype='<U21')

In [12]:
winning_state_txt = np.array(['winning_state'] * len(epi_data))

In [14]:
epi_data = np.hstack((epi_data, winning_state_txt.reshape(-1, 1)))
epi_data = np.hstack((epi_data, bin_argmax_str.reshape(-1, 1)))

In [15]:
epi_data[49]

array(['chr1', '9800', '10000', '0', '0', '0', '0', '0', '0', '0', '0',
       '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 'winning_state',
       'nan'], dtype='<U21')

In [16]:
epi_data[50]

array(['chr1', '10000', '10200', '0.0', '0.00049', '0.0', '0.0', '0.0',
       '-0.00727', '0.0', '0.0', '0.0', '-0.00302', '-0.00389',
       '-0.00446', '-0.01614', '0.01248', '0.0', '-0.00401', '0.0',
       '0.36155', 'winning_state', '17'], dtype='<U21')

In [18]:
epi_data[50, [0, 1, 2, -2, -1]]

array(['chr1', '10000', '10200', 'winning_state', '17'], dtype='<U21')

In [19]:
outfile = '/home/ctrader/region_similarity/data/epilogos_scores_new.winningstate.officialorder.CT20230103.txt'
np.savetxt(outfile, epi_data[:, [0, 1, 2, -2, -1]], delimiter='\t', fmt='%s')

In [ ]:
"""
sort-bed epilogos_scores_new.winningstate.officialorder.CT20230103.txt > epilogos_scores_new.winningstate.officialorder.sorted.CT20230103.bed
bedmap --echo --echo-map-score Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.bed epilogos_scores_new.winningstate.officialorder.sorted.CT20230103.bed > regions_25kb_CT20230103.epilogos.maxstateperbin.officialstateorder.allmapped.bed
"""

Priors

In [10]:
filename = '/home/ctrader/region_similarity/data/regions_25kb_CT20230103.epilogos.maxstateperbin.officialstateorder.allmapped.bed'
f = open(filename, 'r')
winning_state_data = []
for line in f.readlines():
    winning_state_data.append(np.float_(line.strip().split('|')[1].split(';')))
winning_state_data = np.array(winning_state_data)
f.close()

In [12]:
num_states = 18
counts = []
for region in range(len(winning_state_data)):
    region_states = (winning_state_data[region][~np.isnan(winning_state_data[region])]).astype(int)
    state_count = np.bincount(region_states)
    state_count = np.append(state_count, np.int_([0] * (num_states - len(state_count))))
    counts.append(state_count)
counts = np.array(counts)

In [30]:
observed = (counts.T / np.sum(counts, axis=1)).T

/tmp/ipykernel_1569/407407469.py:1: RuntimeWarning: invalid value encountered in true_divide
  observed = (counts.T / np.sum(counts, axis=1)).T


In [31]:
filename = '/home/ctrader/region_similarity/data/rank_based/epilogos.winningstateperbin.genomewide.counts.expectedratio.officialorder.CT20230103.tsv'
f = open(filename, 'r')
expected = np.float_(f.readlines()[2].strip().split('\t'))
f.close()

In [32]:
ooe = observed / expected

In [36]:
ooe_mean = np.nanmean(ooe, axis=1)

/tmp/ipykernel_1569/2155104413.py:1: RuntimeWarning: Mean of empty slice
  ooe_mean = np.nanmean(ooe, axis=1)


In [62]:
def sigmoid_func_prior(val_arr, scaling_factor=5):

    # max rank
    max_val = np.nanmax(val_arr)

    # scaling between [-scaling_factor, scaling_factor]
    x = (val_arr / (max_val / (scaling_factor * 2))) - scaling_factor
    sig_arr = 1 / (1 + np.exp(-x))
    return sig_arr

In [68]:
priors = sigmoid_func_prior(ooe_mean)
priors = np.where(np.isnan(priors), 0, priors)

In [72]:
priors

array([0.00679354, 0.00679354, 0.00679354, ..., 0.        , 0.        ,
       0.        ])

In [74]:
outfile = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20230103.priors.maxstate.epilogos.tsv'
np.savetxt(outfile, priors, delimiter='\t', fmt='%s')

**Priors for fusion score - Epilogos, DHS, TF**

In [2]:
filename = '/home/ctrader/region_similarity/data/epilogos_scores_new_CT20220815.txt'
f = open(filename, 'r')
epi_data = []
for line in f.readlines():
    epi_data.append(line.strip().split('\t'))
epi_data = np.array(epi_data)

In [3]:
epi_data[:, 3:].shape

(15441361, 18)

In [4]:
region_sums = np.sum(epi_data[:, 3:].astype(float), axis=1)

In [5]:
region_sums[:100]

array([0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.33573, 0.34249, 0.34442, 0.31364, 0.32529, 0.32894,
       0.33771, 0.34331, 0.34331, 0.34331, 0.34331, 0.34331, 0.34331,
       0.34331, 0.34331, 0.33949, 0.34442, 0.34442, 0.34119, 0.34119,
       0.34119, 0.34119, 0.34119, 0.34119, 0.34119, 0.34119, 0.34119,
       0.34119, 0.3351 , 0.3351 , 0.33071, 0.28858, 0.28858, 0.29435,
       0.29435, 0.29435, 0.2874 , 0.2874 , 0.29284, 0.29591, 0.29752,
       0.29919, 0.29919, 0.34779, 0.35882, 0.40048, 0.40048, 0.40048,
       0.40048, 0.40

In [6]:
epi_data.shape

(15441361, 21)

In [7]:
region_sums.reshape(-1, 1).shape

(15441361, 1)

In [8]:
np.hstack((region_sums.reshape(-1, 1)[:100], region_sums.reshape(-1, 1)[:100]))

array([[0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
       [0.     , 0.     ],
 

In [10]:
states_sum_txt = np.array(['states_sum'] * len(epi_data))

In [12]:
epi_data = np.hstack((epi_data, states_sum_txt.reshape(-1, 1)))
epi_data = np.hstack((epi_data, region_sums.reshape(-1, 1).astype(str)))

In [45]:
#without the "states_sum" column
outfile = '/home/ctrader/region_similarity/data/epilogos_scores_new_rowsum_CT20221215.txt'
np.savetxt(outfile, epi_data, delimiter='\t', fmt='%s')

In [14]:
outfile = '/home/ctrader/region_similarity/data/epilogos_scores_new_rowsumonly_CT20221215.txt'
np.savetxt(outfile, epi_data[:, [0, 1, 2, -2, -1]], delimiter='\t', fmt='%s')

In [13]:
"""
sort-bed epilogos_scores_new_rowsumonly_CT20221215.txt > epilogos_scores_new_rowsumonly_CT20221215.sorted.txt
bedmap --echo --max Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.bed epilogos_scores_new_rowsumonly_CT20221215.sorted.txt > regions_25kb_CT20221215.epilogos.rowsum.maxmapped.bed
"""

In [26]:
filename = '/home/ctrader/region_similarity/data/regions_25kb_CT20221215.epilogos.rowsum.maxmapped.bed'
f = open(filename, 'r')
epi_maxsum = []
for line in f.readlines():
    epi_maxsum.append(re.split('[\t|]', line.strip())[-1])
epi_maxsum = np.float_(epi_maxsum)

In [31]:
def sigmoid_func_post(val_arr, scaling_factor=5):

    # max rank
    max_val = np.max(val_arr)

    # scaling between [-scaling_factor, scaling_factor]
    x = (val_arr / (max_val / (scaling_factor * 2))) - scaling_factor
    sig_arr = 1 / (1 + np.exp(-x))
    return sig_arr

In [32]:
epi_maxsum_priors = sigmoid_func_post(epi_maxsum)

In [36]:
DHS_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221206.DHSsignalzscores.hdf5'
TF_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221214.TFsignalzscores.hdf5'

#load the data
DHS_z_hdf5 = h5py.File(DHS_base_path, 'r')
DHS_z_arr = DHS_z_hdf5['base_arr'][:]
DHS_z_hdf5.close()

TF_z_hdf5 = h5py.File(TF_base_path, 'r')
TF_z_arr = TF_z_hdf5['base_arr'][:]
TF_z_hdf5.close()

In [44]:
DHS_max_z_arr = np.max(DHS_z_arr, axis=1)
TF_max_z_arr = np.max(TF_z_arr, axis=1)

In [57]:
DHS_min_p_arr = stats.norm.sf(DHS_max_z_arr)
TF_min_p_arr = stats.norm.sf(TF_max_z_arr)

In [62]:
DHS_p_priors = 1 - DHS_min_p_arr
TF_p_priors = 1 - TF_min_p_arr

In [75]:
priors = np.hstack((DHS_p_priors.reshape(-1, 1), TF_p_priors.reshape(-1, 1)))
priors = np.hstack((priors, epi_maxsum_priors.reshape(-1, 1)))

In [80]:
DHS_p_priors

array([0.5, 0.5, 0.5, ..., 0.5, 0.5, 0.5])

In [81]:
TF_p_priors

array([0.99995897, 0.99959253, 0.99977493, ..., 0.5       , 0.5       ,
       0.5       ])

In [82]:
epi_maxsum_priors

array([0.01008324, 0.01008324, 0.01008324, ..., 0.00669285, 0.00669285,
       0.00669285])

In [86]:
regions_path='/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.bed'
f = open(regions_path, 'r')
regions = f.readlines()
f.close()

In [91]:
epi_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogosmeanmax.hdf5'
epi_base_hdf5 = h5py.File(epi_base_path, 'r')
epi_meanmax_arr = epi_base_hdf5['base_arr'][:]
epi_base_hdf5.close()

epi_mean_arr = epi_meanmax_arr[:, :18]
epi_mean_arr = np.where(epi_mean_arr < 0, 0, epi_mean_arr)

In [94]:
epi_sim_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221214.epilogos.mean.sigmoid.conv.sqrt.sigmoid.hdf5'
hdf5_file = h5py.File(epi_sim_path, 'r')
# epi_similar_regions = hdf5_file['top_regions'][:]
epi_sigmoid_scaled_vals = hdf5_file['top_similarity_vals_sqrt_sigmoid'][:]
hdf5_file.close()

In [97]:
DHS_sim_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221214.DHSsignal.z.per.conv.sqrt.sigmoid.hdf5'
hdf5_file = h5py.File(DHS_sim_path, 'r')
# DHS_similar_regions = hdf5_file['top_regions'][:]
DHS_sigmoid_scaled_vals = hdf5_file['top_similarity_vals_sqrt_sigmoid'][:]
hdf5_file.close()

In [98]:
TF_sim_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221214.TFsignal.z.per.conv.sqrt.sigmoid.hdf5'
hdf5_file = h5py.File(TF_sim_path, 'r')
# TF_similar_regions = hdf5_file['top_regions'][:]
TF_sigmoid_scaled_vals = hdf5_file['top_similarity_vals_sqrt_sigmoid'][:]
hdf5_file.close()

In [88]:
x = np.argmax(priors[:, 2])
print(x)
print(priors[x])

39476
[0.99940821 1.         0.99330715]


In [101]:
x = regions[39476].strip().split('\t')
print(x[0] + ':' + x[1] + '-' + x[2])
print(x[0] + ':' + str(int(x[1])+1) + '-' + str(int(x[1])+2))

chr19:3963000-3988000
chr19:3963001-3963002


In [93]:
epi_mean_arr[39476]

array([1.423550e-01, 1.206670e-01, 1.604300e-02, 5.813880e-01,
       6.386140e-01, 1.875400e-02, 0.000000e+00, 0.000000e+00,
       1.546000e-03, 0.000000e+00, 5.743540e-01, 7.467200e-02,
       8.119900e-02, 1.049040e-01, 6.436700e-01, 1.890661e+00,
       4.878180e-01, 0.000000e+00])

In [95]:
epi_sigmoid_scaled_vals[39476]

array([0.41840591, 0.33002604, 0.30901241, 0.3020086 , 0.28975191,
       0.27691896, 0.27599465, 0.27580325, 0.26908249, 0.2663082 ,
       0.26318161, 0.24998029, 0.24476788, 0.2430734 , 0.23600073,
       0.23529813, 0.22678042, 0.21706807, 0.21688225, 0.21419323,
       0.2140811 , 0.21289621, 0.21070871, 0.20946678, 0.20647947,
       0.20614424, 0.20282072, 0.20230093, 0.20116613, 0.20088597,
       0.20043313, 0.20042254, 0.20037738, 0.20002324, 0.19990564,
       0.19969026, 0.19966279, 0.1995538 , 0.19935838, 0.19933055,
       0.19920992, 0.19881821, 0.19880724, 0.19839334, 0.19833927,
       0.1979472 , 0.19777464, 0.19764022, 0.1975838 , 0.19756552,
       0.1975566 , 0.19749878, 0.19736346, 0.1970387 , 0.19680966,
       0.19678144, 0.19657723, 0.1965737 , 0.19600951, 0.19595384,
       0.19585712, 0.19568889, 0.19563439, 0.19544233, 0.19540032,
       0.19493923, 0.19463923, 0.19461326, 0.19446582, 0.19435591,
       0.19425643, 0.19390623, 0.19386281, 0.19378609, 0.19369

In [99]:
DHS_sigmoid_scaled_vals[39476]

array([0.93762117, 0.93571648, 0.93568334, 0.92984835, 0.92914059,
       0.92833357, 0.92763815, 0.92475001, 0.92396063, 0.92333301,
       0.92297453, 0.92068297, 0.9205082 , 0.92007379, 0.91890598,
       0.91764867, 0.91751683, 0.91677819, 0.91368177, 0.91194501,
       0.91128713, 0.91045763, 0.90973651, 0.90934498, 0.9090579 ,
       0.90904361, 0.90904192, 0.90831007, 0.90815098, 0.90731947,
       0.90676653, 0.90596985, 0.90553413, 0.90496458, 0.90467115,
       0.90461936, 0.90425187, 0.90316319, 0.9029697 , 0.902538  ,
       0.90238784, 0.90220473, 0.90200783, 0.90175446, 0.90171547,
       0.90075415, 0.90027018, 0.89990405, 0.89928076, 0.89902523,
       0.89875521, 0.89864934, 0.89832547, 0.89804711, 0.89748841,
       0.89712074, 0.89708144, 0.89689913, 0.89664039, 0.8964429 ,
       0.89614042, 0.89613625, 0.89592242, 0.89463355, 0.89435817,
       0.89429119, 0.89404468, 0.89401958, 0.89399129, 0.89396371,
       0.89330412, 0.89322224, 0.89225028, 0.89189346, 0.89176

In [100]:
TF_sigmoid_scaled_vals[39476]

array([0.99257313, 0.99232762, 0.99221962, 0.99218248, 0.99212512,
       0.9920912 , 0.99206514, 0.99204663, 0.99203764, 0.99202914,
       0.99196382, 0.99195333, 0.99195185, 0.9918919 , 0.99187382,
       0.9918724 , 0.99186568, 0.99183418, 0.99182793, 0.99182532,
       0.9917788 , 0.99176864, 0.99175852, 0.99175212, 0.99174659,
       0.99173507, 0.99173178, 0.9917212 , 0.99167955, 0.99167347,
       0.99166712, 0.99163969, 0.99163793, 0.991594  , 0.99157288,
       0.99155116, 0.99153005, 0.99150965, 0.99149556, 0.99149354,
       0.99145989, 0.99143401, 0.99140461, 0.99137115, 0.99136379,
       0.99136222, 0.99136107, 0.99135536, 0.99134568, 0.99133063,
       0.99132775, 0.99129173, 0.99128819, 0.99128222, 0.99127342,
       0.99126938, 0.99126698, 0.99126269, 0.99125593, 0.99125319,
       0.99123769, 0.99123257, 0.99122363, 0.99121861, 0.99121669,
       0.99120584, 0.99120359, 0.99120091, 0.99119652, 0.99117916,
       0.99117447, 0.99114749, 0.99114677, 0.99114139, 0.99113

In [83]:
outfile = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221215.priors.DHS.TF.epilogos.tsv'
np.savetxt(outfile, priors, delimiter='\t', fmt='%s')

**Split up TF Motif data from JV**

In [10]:
# outdir = '/home/ctrader/region_similarity/data/tf_motif_bed_CT20221213'
filename = '/home/ctrader/region_similarity/data/rank_based/hg38.archetype_motifs.v1.0.bed'
f = open(filename, 'r')

while True:
    line = f.readline()
    if not line:
        break
    print(line)
f.close()

chr1	10005	10022	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10011	10028	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10017	10034	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10023	10040	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10029	10046	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10035	10052	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10041	10058	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10047	10064	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10053	10070	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10059	10076	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10065	10082	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10071	10088	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10077	10094	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10083	10100	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10089	10106	KLF/SP/2	3.7536	-	RREB1_MA0073.1	1

chr1	10095	10112	KLF/SP/2	5.2260	-	RREB1_MA0073.1	1

chr1	10096	10113	NR/12	7.7009	-	RARA+RXRG_MA1149.1	1

chr1	10100	10117	KLF/SP/2	3.8214	-	RREB1_MA0073.1	1

chr1	10101	10118	KLF/SP/2	7.6998	-	RREB1_MA00


chr1	17240	17254	Ebox/CAGATGG	11.6473	-	PTF1A_HUMAN.H11MO.0.B	14

chr1	17241	17248	Ebox/CACCTG	7.5774	-	TCF3_MA0522.2	2

chr1	17245	17254	FEZF1	7.9804	+	FEZF1_HUMAN.H11MO.0.C	1

chr1	17246	17257	PRDM1	7.3532	-	PRDM1_HUMAN.H11MO.0.A	2

chr1	17257	17265	HD/2	7.1775	-	ISL1_HUMAN.H11MO.0.A	3

chr1	17258	17268	HD/12	7.5131	+	PBX1_MOUSE.H11MO.0.A	3

chr1	17259	17273	PLAG1	3.3969	+	PLAG1_MA0163.1	1

chr1	17259	17278	ZNF320	6.8164	+	ZN320_HUMAN.H11MO.0.C	1

chr1	17261	17273	PRDM5	7.7640	+	PRDM5_MOUSE.H11MO.0.A	1

chr1	17261	17275	HIC/1	7.7561	-	HIC2_C2H2_1	2

chr1	17262	17276	CTCF	9.1745	+	CTCFL_MA1102.1	1

chr1	17262	17282	GC-tract	7.2295	+	ZN341_HUMAN.H11MO.0.C	1

chr1	17263	17275	INSM1	6.7102	+	INSM1_HUMAN.H11MO.0.C	2

chr1	17266	17275	ZBTB48	8.1145	+	ZBT48_HUMAN.H11MO.0.C	1

chr1	17273	17287	Ebox/CAGATGG	7.2228	+	PTF1A_HUMAN.H11MO.0.B	2

chr1	17279	17288	Ebox/CAGCTG	8.1384	+	BHA15_HUMAN.H11MO.0.B	4

chr1	17279	17293	Ebox/CAGATGG	7.8814	-	OLIG2_HUMAN.H11MO.0.B	2

chr1	17280	17297	KLF/SP/2	

chr1	28403	28415	E2F/3	8.3661	-	E2F8_E2F_1	4

chr1	28403	28417	FOX/3	4.3007	+	FOXI1_forkhead_2	1

chr1	28406	28414	E2F/2	7.6216	+	E2F7_HUMAN.H11MO.0.B	4

chr1	28408	28423	NFAT/2	7.7463	-	NFATC3_MA0625.1	2

chr1	28411	28418	ZNF384/2	8.2363	+	ZNF384_MA1125.1	1

chr1	28412	28419	ZNF384/2	8.4241	+	ZNF384_MA1125.1	1

chr1	28447	28461	ZNF282	5.1722	-	ZNF282_C2H2_1	2

chr1	28449	28458	HD/17	9.4044	-	HOXA13_homeodomain_4	13

chr1	28449	28458	HD/18	9.0222	-	HOXD13_MA0909.1	11

chr1	28451	28460	STAT/2	7.7683	+	Stat4_MA0518.1	2

chr1	28468	28484	IRF/2	7.1733	-	BC11A_HUMAN.H11MO.0.A	1

chr1	28474	28489	NFAT/3	6.5740	-	NFATC1_NFAT_2	2

chr1	28484	28499	NR/17	5.1697	-	PPARG_MA0066.1	1

chr1	28484	28500	NR/16	7.5874	+	Nr1h3+Rxra_MA0494.1	1

chr1	28485	28494	OVOL1	9.4746	+	OVOL1_HUMAN.H11MO.0.C	2

chr1	28487	28500	SOX/3	8.3080	-	Sox17.mouse_HMG_2	15

chr1	28487	28501	SOX/5	1.1578	-	SOX8_HMG_4	3

chr1	28506	28520	HAND1	7.4038	+	HAND1_MOUSE.H11MO.0.C	1

chr1	28512	28537	ZNF354	7.7784	+	ZN394_HUMAN.H11MO


chr1	39865	39879	MTF1	5.8031	+	MTF1_HUMAN.H11MO.0.C	2

chr1	39868	39876	HD/22	8.8783	+	NKX25_MOUSE.H11MO.0.A	7

chr1	39868	39881	TEAD	7.2119	+	TEAD2_MOUSE.H11MO.0.C	1

chr1	39875	39892	ZNF332	8.0147	-	ZN322_MOUSE.H11MO.0.B	2

chr1	39878	39897	ZFN121	7.1569	+	ZN121_HUMAN.H11MO.0.C	1

chr1	39879	39889	HD/12	9.9387	+	PBX3_MA1114.1	3

chr1	39880	39897	PAX/2	11.1269	+	PAX5_HUMAN.H11MO.0.A	2

chr1	39881	39893	FOX/9	7.3163	+	FOXB1_forkhead_1	1

chr1	39882	39893	AP1/2	7.6485	-	BACH2_MOUSE.H11MO.0.A	2

chr1	39905	39912	ZNF384/2	8.0452	+	ZNF384_MA1125.1	1

chr1	39906	39918	FOX/5	9.8321	+	FOXJ3_HUMAN.H11MO.0.A	8

chr1	39907	39917	FOX/4	7.8231	-	FOXC2_MA0846.1	2

chr1	39908	39917	CUX/1	7.8838	+	ONECUT3_CUT_1	2

chr1	39910	39922	FOX/5	11.0274	+	FOXC1_forkhead_1	9

chr1	39911	39921	FOX/4	8.0749	-	FOXC2_MA0846.1	4

chr1	39911	39923	POU/2	7.0685	+	POU3F1_POU_2	1

chr1	39914	39926	FOX/5	7.4284	+	FOXC1_forkhead_1	1

chr1	39929	39941	NR/4	3.9708	-	Rxra.mouse_nuclearreceptor_2	4

chr1	39936	39948	GRHL	10

chr1	62589	62602	SOX/1	7.6764	+	SOX17_HUMAN.H11MO.0.C	1

chr1	62592	62606	NR/3	7.3008	-	NR4A1_MOUSE.H11MO.0.B	1

chr1	62603	62612	DDIT3+CEBPA	8.1702	-	Ddit3+Cebpa_MA0019.1	1

chr1	62607	62618	SIX/1	7.6699	-	SIX1_HUMAN.H11MO.0.A	1

chr1	62612	62623	ETS/2	7.8160	-	ELF3_HUMAN.H11MO.0.A	2

chr1	62612	62624	SPI	7.2718	-	SPIC_ETS_1	2

chr1	62617	62630	SOX/1	7.6341	+	SRY_HUMAN.H11MO.0.B	2

chr1	62617	62636	ZNF680	3.2780	-	ZN680_HUMAN.H11MO.0.C	1

chr1	62618	62643	ZNF354	7.8507	+	Z354A_HUMAN.H11MO.0.C	1

chr1	62619	62631	FOX/5	8.4424	-	FOXJ3_HUMAN.H11MO.0.A	2

chr1	62621	62633	FOX/5	9.1001	-	Foxj3.mouse_forkhead_4	4

chr1	62624	62636	FOX/5	6.9985	-	FOXJ3_forkhead_1	2

chr1	62633	62646	SRF	6.3121	-	SRF_MADS_1	1

chr1	62634	62647	SRF	10.5688	+	SRF_MOUSE.H11MO.0.A	2

chr1	62639	62653	E2F/1	2.9330	-	E2F3_E2F_1	2

chr1	62640	62652	E2F/3	1.5782	-	E2F7_E2F_1	2

chr1	62656	62681	ZNF354	7.2139	+	ZN394_HUMAN.H11MO.0.C	1

chr1	62663	62682	SMARCA5	7.5943	-	SMCA5_MOUSE.H11MO.0.C	1

chr1	62668	62684	IRF/2	8

...

chr1	97442	97454	IRF/1	8.4738	-	IRF2_MA0051.1	5

chr1	97442	97456	SPDEF/2	5.9736	-	SPDEF_ETS_3	1

chr1	97442	97458	IRF/2	6.9213	-	IRF1_MA0050.2	1

chr1	97460	97468	HD/2	8.1017	+	Hoxd8_MA0910.1	3

chr1	97461	97477	HD/5	5.8182	-	MSX2_homeodomain_1	1

chr1	97468	97476	HD/2	4.2805	+	POU6F2_POU_2	1

chr1	97468	97480	POU/2	7.1203	+	POU4F1_MA0790.1	2

chr1	97471	97480	HD/18	6.4682	+	Arid3a_MA0151.1	1

chr1	97479	97500	ZNF382	6.0716	+	ZN382_HUMAN.H11MO.0.C	1

chr1	97486	97501	MAF	9.0100	+	MAF_MOUSE.H11MO.0.A	8

chr1	97489	97508	ZNF547	8.6106	+	ZN547_HUMAN.H11MO.0.C	1

chr1	97490	97499	CREB/ATF/1	7.7278	+	ATF2_HUMAN.H11MO.0.B	2

chr1	97490	97500	CREB/ATF/2	7.5611	+	JUN_MA0488.1	1

chr1	97490	97500	CREB/ATF/3	8.6197	+	CEBPG_HUMAN.H11MO.0.B	7

chr1	97494	97505	NR/19	7.3486	-	NR1D1_HUMAN.H11MO.0.B	2

chr1	97495	97505	SCRT1	5.3968	+	SCRT1_C2H2_1	2

chr1	97497	97506	Ebox/CAGCTG	7.5981	-	MYF6_MA0667.1	2

chr1	97498	97507	Ebox/CAGCTG	6.7809	+	MYOG_HUMAN.H11MO.0.B	2

chr1	97507	97517	GATA	9.6419	+	GATA


chr1	121311	121326	ZSCAN3	8.1181	+	ZSC31_HUMAN.H11MO.0.C	1

chr1	121313	121321	HD/22	8.8783	-	NKX25_MOUSE.H11MO.0.A	7

chr1	121318	121327	Ebox/CAGCTG	7.6490	-	ZBT18_HUMAN.H11MO.0.C	1

chr1	121325	121341	NR/16	7.6870	+	THA_HUMAN.H11MO.0.C	1

chr1	121327	121347	GC-tract	8.4558	-	THAP1_HUMAN.H11MO.0.C	1

chr1	121333	121342	SREBF1	9.6420	-	SRBP1_MOUSE.H11MO.0.B	3

chr1	121337	121357	GC-tract	7.8479	-	TAF1_HUMAN.H11MO.0.A	3

chr1	121339	121349	RUNX/1	7.9297	-	RUNX2_MA0511.2	4

chr1	121339	121356	KLF/SP/2	7.1321	-	KLF6_HUMAN.H11MO.0.A	2

chr1	121340	121357	KLF/SP/2	8.1650	-	PATZ1_HUMAN.H11MO.0.C	1

chr1	121345	121362	KLF/SP/2	9.7244	-	SP2_HUMAN.H11MO.0.A	7

chr1	121346	121358	ZBTB6	9.3978	-	ZBTB6_HUMAN.H11MO.0.C	1

chr1	121347	121359	PRDM5	7.3643	-	PRDM5_MOUSE.H11MO.0.A	1

chr1	121347	121361	CTCF	8.5006	-	CTCFL_HUMAN.H11MO.0.A	2

chr1	121347	121366	ZNF143	6.6863	-	ZNF76_HUMAN.H11MO.0.C	5

chr1	121351	121366	NR/15	7.9753	+	VDR_MA0693.2	1

chr1	121354	121371	KLF/SP/2	7.5044	-	KLF6_HUMAN.H11MO

chr1	135901	135912	SPZ1	7.7008	-	Spz1_MA0111.1	1

chr1	135901	135918	ZNF708	9.3602	-	ZN708_HUMAN.H11MO.0.C	1

chr1	135902	135913	ETS/2	6.8508	-	ELF5_MOUSE.H11MO.0.A	1

chr1	135903	135923	GC-tract	8.5956	-	ZN341_HUMAN.H11MO.0.C	1

chr1	135904	135918	SPDEF/2	7.8075	-	SPDEF_ETS_6	1

chr1	135904	135918	CTCF	7.0279	-	CTCFL_HUMAN.H11MO.0.A	2

chr1	135904	135921	PAX/2	8.0162	-	PAX5_MOUSE.H11MO.0.A	1

chr1	135904	135924	GC-tract	7.9805	-	ZN770_HUMAN.H11MO.0.C	1

chr1	135909	135922	RFX/3	8.2193	-	RFX5_HUMAN.H11MO.0.A	1

chr1	135911	135925	NFI/1	9.6598	+	NFIB_MOUSE.H11MO.0.C	2

chr1	135911	135931	GC-tract	10.6001	-	MAZ_HUMAN.H11MO.0.A	3

chr1	135912	135929	KLF/SP/2	5.4137	-	SP1_MOUSE.H11MO.0.A	1

chr1	135913	135930	KLF/SP/2	7.5209	-	PATZ1_HUMAN.H11MO.0.C	1

chr1	135914	135930	MZF1	6.4097	-	ZN816_HUMAN.H11MO.0.C	1

chr1	135914	135933	ZNF143	8.5821	-	ZNF76_HUMAN.H11MO.0.C	5

chr1	135914	135934	GC-tract	8.7064	-	ZN263_HUMAN.H11MO.0.A	3

chr1	135914	135937	TBX/4	7.7123	-	TBX15_MA0803.1	4

chr1	13591

chr1	140546	140560	ZIM3	7.9136	-	ZIM3_HUMAN.H11MO.0.C	1

chr1	140555	140571	NR/5	3.3107	-	THRB_nuclearreceptor_1	1

chr1	140560	140567	Ebox/CACCTG	8.3196	+	HTF4_MOUSE.H11MO.0.A	3

chr1	140560	140569	Ebox/CAGCTG	9.2555	+	TWIST1_MA1123.1	4

chr1	140560	140569	Ebox/CATATG	7.1266	-	Neurog1_MA0623.1	1

chr1	140560	140574	Ebox/CAGATGG	10.5377	-	NEUROD1_MA1109.1	9

chr1	140564	140583	ZFN121	8.7220	+	ZN121_HUMAN.H11MO.0.C	1

chr1	140565	140576	NR/19	7.5727	-	NR1D1_HUMAN.H11MO.0.B	2

chr1	140567	140578	TFAP2/2	5.9288	-	TFAP2C_TFAP_5	1

chr1	140568	140587	ZNF143	4.4952	-	THA11_MOUSE.H11MO.0.B	1

chr1	140570	140581	YY1	8.9686	+	YY1_MA0095.2	3

chr1	140571	140587	P53-like/1	7.7237	+	P73_HUMAN.H11MO.0.A	2

chr1	140574	140583	ZNF449	7.8671	-	ZN449_HUMAN.H11MO.0.C	1

chr1	140580	140592	PRDM5	7.5780	-	PRDM5_MOUSE.H11MO.0.A	1

chr1	140580	140593	RFX/3	7.4564	-	RFX6_MOUSE.H11MO.0.C	1

chr1	140582	140601	ZFN121	9.6046	+	ZN121_HUMAN.H11MO.0.C	1

chr1	140584	140601	PAX/2	7.9777	+	PAX5_HUMAN.H11MO.0.A	1

ch

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [11]:
line

'chr1\t185183\t185202\tZNF320\t5.5470\t+\tZN320_HUMAN.H11MO.0.C\t1\n'

In [4]:
# outdir = '/home/ctrader/region_similarity/data/tf_motif_bed_CT20221213'
filename = '/home/ctrader/region_similarity/data/rank_based/hg38.archetype_motifs.v1.0.bed'
f = open(filename, 'r')

tf_motifs = []
current_chrm = ''
while True:
    line = f.readline()
    if not line:
        break
    chrm = line.strip().split('\t')[0]
    if chrm != current_chrm:
        print(chrm)
        current_chrm = chrm
    motif = line.strip().split('\t')[3]
    if motif not in tf_motifs:
        tf_motifs.append(motif)
f.close()

chr1
chr10
chr11
chr11_KI270721v1_random
chr12
chr13
chr14
chr14_GL000009v2_random
chr14_GL000194v1_random
chr14_GL000225v1_random
chr14_KI270722v1_random
chr14_KI270723v1_random
chr14_KI270724v1_random
chr14_KI270725v1_random
chr14_KI270726v1_random
chr15
chr15_KI270727v1_random
chr16
chr16_KI270728v1_random
chr17
chr17_GL000205v2_random
chr17_KI270729v1_random
chr17_KI270730v1_random
chr18
chr19
chr1_KI270706v1_random
chr1_KI270707v1_random
chr1_KI270708v1_random
chr1_KI270709v1_random
chr1_KI270710v1_random
chr1_KI270711v1_random
chr1_KI270712v1_random
chr1_KI270713v1_random
chr1_KI270714v1_random
chr2
chr20
chr21
chr22
chr22_KI270731v1_random
chr22_KI270732v1_random
chr22_KI270733v1_random
chr22_KI270734v1_random
chr22_KI270735v1_random
chr22_KI270736v1_random
chr22_KI270737v1_random
chr22_KI270738v1_random
chr22_KI270739v1_random
chr2_KI270715v1_random
chr2_KI270716v1_random
chr3
chr3_GL000221v1_random
chr4
chr4_GL000008v2_random
chr5
chr5_GL000208v1_random
chr6
chr7
chr8
chr9
chr

In [12]:
line

'chr1\t185183\t185202\tZNF320\t5.5470\t+\tZN320_HUMAN.H11MO.0.C\t1\n'

In [8]:
' '.join(tf_motifs)

'KLF/SP/2 NR/12 INSM1 NR/1 NR/3 ZNF282 SPZ1 NR/11 GC-tract RUNX/1 ZNF449 ZNF768 KAISO ZNF324 GCM PAX/2 CTCF ZNF547 NR/16 TFAP2/1 ZNF436 SREBF1 MBD2 NR/18 E2F/2 GLI KLF/SP/1 NFI/3 ZIC/2 NRF1 EGR THAP1 TFAP2/2 NR/17 ZFX ZNF524 GFI ZNF140 HEN1 Ebox/CAGCTG Ebox/CAGATGG ZFN121 MECP2 RFX/3 RFX/2 RFX/1 E2F/4 ZNF332 ZNF134 ZBTB49 ZNF335 P53-like/1 P53-like/3 AP1/2 Ebox/CACGTG/2 ZIC ZNF53 CREB3/XBP1 ETS/1 TBX/4 Ebox/CACGTG/1 PLAG1 MYB/5 ZNF320 NR/15 ZBTB7A LEF1 TBX/2 ZNF554 PRDM5 TBX/3 MFZ1 HIF ZSCAN4 AHR CREB/ATF/1 CREB/ATF/2 PAX-halfsite GATA ZNF306 BATF HD/20 CUX/2 ZBTB48 ZNF563 SMARCA5 HIC/1 ZSCAN3 ZNF667 ZNF329 MTF1 NFI/1 MAF E2F/1 KLF/SP/3 GLIS Ebox/CACCTG NR/19 ZNF85 REST/NRSF ZNF257 ZNF143 ZNF331 MZF1 SMAD HINFP1/2 TEAD HINFP1/1 PRDM9 SNAI2 REL-halfsite SPDEF/1 FOX/4 NFKB/1 NFKB/2 Ebox/CATATG SRF ZNF136 NR/2 FOX/5 FOX/3 FOX/2 HD/18 HD/2 HD/1 OCT4+SOX2 BCL6/2 NFAC/2 E2F/3 FOX/7 GRHL HAND1 ZNF384/2 NFAT/2 TCF/LEF ZNF708 SIX/2 IRF/2 SPI ZNF354 ZNF28 FOX/1 ETS/2 MIES IRF/1 NR/13 HD/22 SIX/1

**Number of connections above STRING threshold**

In [2]:
filename = '/home/ctrader/region_similarity/data/9606.protein.links.full.v11.5.txt'
f = open(filename, 'r')
header = f.readline().strip().split(' ')
f.close()
string_genes_arr = np.loadtxt(filename, dtype=str, skiprows=1, usecols=[0, 1])
string_vals_arr = np.loadtxt(filename, dtype=int, skiprows=1, usecols=np.arange(2, len(header)))

In [3]:
val_header = header[2:]

In [4]:
np.sort(val_header)

array(['coexpression', 'coexpression_transferred', 'combined_score',
       'cooccurence', 'database', 'database_transferred', 'experiments',
       'experiments_transferred', 'fusion', 'homology', 'neighborhood',
       'neighborhood_transferred', 'textmining', 'textmining_transferred'],
      dtype='<U24')

In [5]:
neigh = [0, 1]
fusion = [2]
cooccur = [3]
homology = [4]
coex = [5, 6]
exper = [7, 8]
database = [9, 10]
textmin = [11, 12]
combined_score = [13]
metric_inds = [coex, cooccur, database, exper, fusion, homology, neigh, textmin, combined_score]

In [6]:
unique_string_genes = np.unique(string_genes_arr)

In [137]:
#indicies for each gene in alphabetical order
protein2ind = {}
for gene, ind in zip(unique_string_genes, np.arange(len(unique_string_genes))):
    protein2ind[gene] = ind

In [139]:
#gene ids -> indices
string_gene_inds_arr = np.array(list(map(protein2ind.get, string_genes_arr.flatten())), 
                                dtype=int).reshape(string_genes_arr.shape)

In [12]:
#for each metric, populate [metric, protein1, protein2] array
metric_protein1_protein2 = []
for col_inds in metric_inds:
    print(np.array(val_header)[col_inds])
    #combining human and transferred scores if applicable
    val_arr = np.max(string_vals_arr[:, col_inds], axis=1)
    metric_sparse = sp.sparse.coo_matrix((val_arr, 
                                          (string_gene_inds_arr[:, 0], string_gene_inds_arr[:, 1])))
    metric_protein1_protein2.append(metric_sparse.toarray())
metric_protein1_protein2 = np.array(metric_protein1_protein2)

['coexpression' 'coexpression_transferred']
['cooccurence']
['database' 'database_transferred']
['experiments' 'experiments_transferred']
['fusion']
['homology']
['neighborhood' 'neighborhood_transferred']
['textmining' 'textmining_transferred']
['combined_score']


In [27]:
metric_protein1_protein2.shape

(9, 19385, 19385)

In [17]:
metric_order = ['coexpression', 'cooccurence', 'database', 'experiments', 
                'fusion', 'homology', 'neighborhood', 'textmining', 'combined_score']

In [36]:
outpath = '/home/ctrader/region_similarity/data/rank_based/9606.protein.links.full.v11.5.allMetrics.proteinXProteinSTRINGScore.hdf5'
out_f = h5py.File(outpath, 'w')
dset = out_f.create_dataset('metric_protein1_protein2', data=metric_protein1_protein2)
dset = out_f.create_dataset('metric_order', data=np.char.encode(metric_order))
dset = out_f.create_dataset('protein_order', data=np.char.encode(unique_string_genes))
out_f.close()

In [78]:
unique_string_genes

array(['9606.ENSP00000000233', '9606.ENSP00000000412',
       '9606.ENSP00000001008', ..., '9606.ENSP00000485668',
       '9606.ENSP00000485672', '9606.ENSP00000485678'], dtype='<U20')

In [ ]:
#for each region, find the number of interconnections above some threshold

In [83]:
#load mapped gene ids
gene_id_mapped_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221027.proteinidmapped.bed'
f = open(gene_id_mapped_path, 'r')
all_regions_gene_ids = []
unique_string_genes_set = set(unique_string_genes)
for line in f.readlines():
    region_gene_ids = []
    geneids = re.split('\t|;|\|', line.strip())[3:]
    for geneid in geneids:
        geneid = '9606.' + geneid.split('.')[0] #removing everything after ., adding 9606.
        if geneid in unique_string_genes_set:
            region_gene_ids.append(geneid)
    all_regions_gene_ids.append(list(np.unique(region_gene_ids)))
f.close()

In [283]:
def rank_regions(arr):
    #rank each region
    ranked_regions = rankdata(arr, axis=0)

    #find the number of regions with 0 values for each component
    comp_num_zero = len(arr) - np.count_nonzero(arr, axis=0)

    #adjusting/resetting ranks
    ranked_regions = ranked_regions - comp_num_zero
    ranked_regions = np.where(ranked_regions < 0, 0, ranked_regions)

    return ranked_regions

In [290]:
for thresh in [0, 100, 200, 300, 400, 500, 600, 700, 800, 900]:
    print(thresh)
    all_regions_counts_per_metric = []
    for region in range(len(all_regions_gene_ids)):
        top_protein_ind = ''
        top_protein_ids_meeting_thresh = np.array([])
        #finding top gene for region
        for protein_id in all_regions_gene_ids[region]:
            protein_ind = protein2ind[protein_id]
            combined_scores = metric_protein1_protein2[-1, protein_ind, :]
            protein_ids_meeting_thresh = np.arange(len(combined_scores))[combined_scores > thresh]
            if len(protein_ids_meeting_thresh) > len(top_protein_ids_meeting_thresh): 
                top_protein_ind = protein_ind
                top_protein_ids_meeting_thresh = protein_ids_meeting_thresh
        #for best gene/protein, find counts from each metric
        if top_protein_ind != '':
            top_protein_arr = metric_protein1_protein2[:, top_protein_ind, top_protein_ids_meeting_thresh]
            counts_for_region = np.count_nonzero(top_protein_arr, axis=1)
        else:
            counts_for_region = np.zeros(len(metric_protein1_protein2), dtype=int)
        all_regions_counts_per_metric.append(counts_for_region)
    all_regions_counts_per_metric = np.array(all_regions_counts_per_metric)
    
    #ranking regions
    ranked_regions = rank_regions(all_regions_counts_per_metric)
    
    #max ranks for each individual STRING metric
    max_ranks = np.max(ranked_regions, axis=0)

    outpath = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221115.top_gene.combined_scores_above_' + str(thresh) + '.counts_by_metric.hdf5'
    out_f = h5py.File(outpath, 'w')
    dset = out_f.create_dataset('all_regions_counts_per_metric', data=all_regions_counts_per_metric)
    dset = out_f.create_dataset('ranked_regions', data=ranked_regions)
    dset = out_f.create_dataset('max_ranks', data=max_ranks)
    dset = out_f.create_dataset('comp_order', data=np.char.encode(metric_order))
    out_f.close()

0
100
200
300
400
500
600
700
800
900


In [293]:
np.max(ranked_regions, axis=0)

array([32844.5,  2831. , 29478.5, 32906.5,  1315.5, 11757.5,  4002. ,
       37032.5, 37216.5])

In [291]:
max_ranks

array([32844.5,  2831. , 29478.5, 32906.5,  1315.5, 11757.5,  4002. ,
       37032.5, 37216.5])

In [289]:
rank_regions(all_regions_counts_per_metric)

array([[    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       [    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       [    0. ,     0. ,     0. , ...,     0. ,     0. ,     0. ],
       ...,
       [27837. ,     0. , 22719. , ...,     0. , 29961. , 29665. ],
       [27837. ,     0. , 22719. , ...,     0. , 29961. , 29665. ],
       [ 7148. ,     0. ,  3950.5, ...,     0. ,  6765.5,  6784.5]])

**Gencode file with expanded regulatory domains (as in GREAT)**

In [4]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.gtf', 'r')
gencode_file = f.readlines()
f.close()

In [5]:
data = []
for line in gencode_file[5:]:
    line = re.split('\t|;| ', line.strip())
    #gene coordinates
    gene_chrm = line[0]
    gene_start = line[3]
    gene_end = line[4]
    gene_strand = line[6]
    gene_type = line[2]
    
    #gene name
    gene_name_index = line.index('gene_name')
    gene_name = line[gene_name_index + 1].strip("\"")
    
    #gene id (ENSG)
    gene_id_index = line.index('gene_id')
    gene_id = line[gene_id_index + 1].strip("\"")
    #protein id (ENSP)
    if 'protein_id' in line:
        protein_id_index = line.index('protein_id')
        protein_id = line[protein_id_index + 1].strip("\"")
    else:
        protein_id = ''
    #promoter start and end
    if gene_strand == '+':
        promoter_start = gene_start
        promoter_end = str(int(gene_start) + 1)
        basal_start = str(int(gene_start) - 5000)
        basal_end = str(int(gene_start) + 1000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    elif gene_strand == '-':
        promoter_start = str(int(gene_end) - 1)
        promoter_end = gene_end
        basal_start = str(int(gene_end) - 1000)
        basal_end = str(int(gene_end) + 5000)
        domain_start = str(int(gene_start) - 1000000)
        domain_end = str(int(gene_end) + 1000000)
        
    data_row = [gene_chrm, gene_start, gene_end, gene_name, gene_type, gene_strand, 
                promoter_start, promoter_end, basal_start, basal_end, domain_start, 
                domain_end, gene_id, protein_id]
    data.append(data_row)

data = np.array(data)

header = ['chrm', 'start', 'end', 'name', 'type', 'strand', 'Promoter start (bp)', 
          'Promoter end (bp)', 'Basal start (bp)', 'Basal end (bp)', 'Domain start (bp)', 
          'Domain end (bp)', 'gene id', 'protein id']

In [6]:
data.shape

(1968859, 14)

In [7]:
#limiting domain to real chromosome positions
chrm = data[:, 0]
tss = data[:, 6].astype(int)
domain_m = data[:, 8:12].astype(int)
names = data[:, 3]

domain_m = np.where(domain_m < 1, 1, domain_m)

In [8]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [9]:
#creating chromosome masked data
chrm_dom_dict = {}
chrm_tss_dict = {}
chrm_name_dict = {}

for chromosome in np.unique(chrm):
    chrm_mask = chrm==chromosome
    chrm_doms = domain_m[chrm_mask, :]
    chrm_tss = tss[chrm_mask]
    chrm_names = names[chrm_mask]
    
    chrm_dom_dict[chromosome] = chrm_doms
    chrm_tss_dict[chromosome] = chrm_tss
    chrm_name_dict[chromosome] = chrm_names

In [43]:
chrm_dom_dict['chr1'].shape

(191060, 4)

In [8]:
chrm_name_dict['chr2']

array(['FAM110C', 'FAM110C', 'FAM110C', ..., 'RPL23AP88', 'RPL23AP88',
       'RPL23AP88'], dtype='<U25')

In [10]:
domain_m

array([[   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1014409],
       [   6869,   12869,       1, 1012227],
       ...,
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023],
       [  15023,   21023,       1, 1016023]])

In [14]:
gene_dom.shape

(4,)

In [25]:
gene

5655

In [47]:
for gene in range(len(domain_m))[191058:191062]:
    if gene%50000==0:
        print(gene)
    
    gene_chrm = chrm[gene]
    gene_tss = tss[gene]
    gene_name = names[gene]
    gene_dom = domain_m[gene, :]
    
    #filtering out wrong chromosomes
#     chrm_mask = chrm==gene_chrm
#     chrm_doms = domain_m[chrm_mask, :]
#     chrm_tss = tss[chrm_mask]
#     chrm_names = names[chrm_mask]
    
    chrm_doms = chrm_dom_dict[gene_chrm]
    chrm_tss = chrm_tss_dict[gene_chrm]
    chrm_names = chrm_name_dict[gene_chrm]
    print(gene_chrm)
    print(chrm_doms.shape)
    print()
    
    #filtering out other annotations of the same gene
    name_mask = chrm_names!=gene_name
    chrm_name_doms = chrm_doms[name_mask, :]
    chrm_name_tss = chrm_tss[name_mask]
    
    #find all the genes to the left and right
    left_genes = chrm_name_tss<gene_tss
    right_genes = chrm_name_tss>gene_tss
    left_doms = chrm_name_doms[left_genes, :]
    right_doms = chrm_name_doms[right_genes, :]

    #if there are any genes, find the closest basal region
    if len(left_doms) > 0:
        max_left_basal_end = np.max(left_doms[:, 1])
    else:
        max_left_basal_end = gene_dom[2]
    if len(right_doms) > 0:
        min_right_basal_start = np.min(right_doms[:, 0])
    else:
        min_right_basal_start = gene_dom[3]

    #setting the domain for each gene
    gene_dom[2] = np.min([np.max([max_left_basal_end, gene_dom[2]]), gene_dom[0]])
    gene_dom[3] = np.max([np.min([min_right_basal_start, gene_dom[3]]), gene_dom[1]])

chr1
(191060, 4)

chr1
(191060, 4)

chr2
(147486, 4)

chr2
(147486, 4)



In [18]:
domain_m[:gene]

array([[   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       [   6869,   12869,       1,   13501],
       ...,
       [2493479, 2499479, 2493479, 2499479],
       [2492853, 2498853, 2492853, 2498853],
       [2525597, 2531597, 2506211, 2531597]])

In [ ]:
#changing domain values
data[:, 8:12] = domain_m.astype(str)

In [15]:
'\t'.join(header)

'chrm\tstart\tend\tname\ttype\tstrand\tPromoter start (bp)\tPromoter end (bp)\tBasal start (bp)\tBasal end (bp)\tDomain start (bp)\tDomain end (bp)\tgene id\tprotein id'

In [ ]:
# outfile = '/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt'
# np.savetxt(outfile, data, delimiter='\t', fmt='%s', header='\t'.join(header))

In [31]:
f = open('/home/ctrader/region_similarity/data/gencode.v41.basic.annotation.parsed.promoterbp.extendeddomain.CT20221026.txt', 'r')
new_gencode_file = f.readlines()
f.close()

In [15]:
new_gencode_file[1]

'chr1\t11869\t14409\tDDX11L1\tgene\t+\t11869\t11870\t6869\t12869\t1\t13501\tENSG00000223972.5\tNA\n'

In [16]:
line.split('\t')

AttributeError: 'list' object has no attribute 'split'

In [32]:
new_data = []
for line in new_gencode_file[1:]:
    new_data.append(line.strip().split('\t'))
new_data = np.array(new_data)

In [49]:
np.unique(new_data[:, 4])

array(['CDS', 'Selenocysteine', 'UTR', 'exon', 'gene', 'start_codon',
       'stop_codon', 'transcript'], dtype='<U25')

In [33]:
np.sum(new_data[:, 8:12][:gene].astype(int) - domain_m[:gene])

0